In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# 0) Imports
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from pyspark.sql import functions as F

# 1) Configurações
BASE = "https://erddap.emodnet.eu"

RELATIVE_URL = (
    "/erddap/griddap/V2023_chemistry_15b1_f0ff_97ab.csv?Water_body_phosphate_deepest%5B(2014-02-16T00:00:00Z):1:(2019-11-16)%5D%5B(36.25):1:(42.0)%5D%5B(-2.25):1:(4.625)%5D"
)

URL = f"{BASE}{RELATIVE_URL}"

TARGET_TABLE = "emodnet_phosphate_deepest"  # nome da tabela no Lakehouse

# Variável principal (como aparece no CSV)
VAR_NAME = "Water_body_phosphate_deepest"
VAR_OUT_COL = "PHO_deepest"  # nome amigável na tabela final

print("Fetching:", URL)

# 2) Leitura do CSV do ERDDAP (ignora metadados começando com '#')
df = pd.read_csv(URL, comment="#")

print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())

# 3) Normalização de colunas (remove aspas e espaços)
df.columns = [c.strip().replace('"', "") for c in df.columns]

# Identifica colunas essenciais
if "time" not in df.columns:
    time_candidates = [c for c in df.columns if c.lower().startswith("time")]
    if not time_candidates:
        raise ValueError(f"Não encontrei coluna de tempo. Colunas: {df.columns.tolist()}")
    df = df.rename(columns={time_candidates[0]: "time"})

if "latitude" not in df.columns or "longitude" not in df.columns:
    raise ValueError(f"Não encontrei latitude/longitude. Colunas: {df.columns.tolist()}")

if VAR_NAME not in df.columns:
    raise ValueError(f"Não encontrei a variável {VAR_NAME}. Colunas: {df.columns.tolist()}")

# Renomeia variável para coluna amigável
df = df.rename(columns={VAR_NAME: VAR_OUT_COL})

# 4) Tipagem e limpeza
# Parse de tempo (sem warning)
df["time"] = pd.to_datetime(
    df["time"],
    format="%Y-%m-%dT%H:%M:%SZ",
    errors="coerce",
    utc=True
)

# Numéricos
for c in ["latitude", "longitude", VAR_OUT_COL]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Remove linhas inválidas
df = df.dropna(subset=["time", "latitude", "longitude", VAR_OUT_COL]).copy()

# Adiciona colunas úteis
df["year"] = df["time"].dt.year.astype(int)
df["ingested_at_utc"] = datetime.now(timezone.utc).isoformat()

print("Clean shape:", df.shape)
df.head()

# 5) Pandas -> Spark
sdf = spark.createDataFrame(df)

# (Opcional) Deduplicar (por tempo+lat+lon)
sdf = sdf.dropDuplicates(["time", "latitude", "longitude"])

# 6) Grava no Lakehouse
# Como é dataset com data fixa, recomendo OVERWRITE para evitar duplicação.
(
    sdf.write
      .mode("overwrite")   # recomendado para snapshot estático
      .format("delta")
      .saveAsTable(TARGET_TABLE)
)

print(f"✅ Gravado no Lakehouse: {TARGET_TABLE}")

# 7) Checagem rápida
spark.table(TARGET_TABLE).orderBy(F.col("latitude").asc(), F.col("longitude").asc()).show(10, truncate=False)


StatementMeta(, 1e59e281-d3c4-474c-ba43-bae8faa8b0b1, 3, Finished, Available, Finished, False)

Fetching: https://erddap.emodnet.eu/erddap/griddap/V2023_chemistry_15b1_f0ff_97ab.csv?Water_body_phosphate_deepest%5B(2014-02-16T00:00:00Z):1:(2019-11-16)%5D%5B(36.25):1:(42.0)%5D%5B(-2.25):1:(4.625)%5D
Raw shape: (63169, 4)
Columns: ['time', 'latitude', 'longitude', 'Water_body_phosphate_deepest']
Clean shape: (38976, 6)
✅ Gravado no Lakehouse: emodnet_phosphate_deepest
+-------------------+--------+---------+-----------+----+--------------------------------+
|time               |latitude|longitude|PHO_deepest|year|ingested_at_utc                 |
+-------------------+--------+---------+-----------+----+--------------------------------+
|2018-08-16 00:00:00|36.25   |-2.25    |0.38515034 |2018|2026-03-31T12:22:07.348031+00:00|
|2017-05-16 00:00:00|36.25   |-2.25    |0.23632331 |2017|2026-03-31T12:22:07.348031+00:00|
|2019-11-16 00:00:00|36.25   |-2.25    |0.39251104 |2019|2026-03-31T12:22:07.348031+00:00|
|2019-02-16 00:00:00|36.25   |-2.25    |0.41455048 |2019|2026-03-31T12:22:07.348